# Ch 6. Neo4j & Cypher — hands-on

지식 그래프와 GraphRAG 스터디 · [study-graphrag](https://desty.github.io/study-graphrag/)

이 노트북은 본문 [Ch 6](https://desty.github.io/study-graphrag/part3/06-neo4j-cypher/)의 코드를 실행 가능한 셀로 옮긴 것입니다. Neo4j 인스턴스(로컬 도커 또는 Aura 무료)가 필요합니다.

In [ ]:
!pip -q install neo4j
from neo4j import GraphDatabase

# Aura를 쓰면 URI/비밀번호를 본인 것으로 교체
URI = "bolt://localhost:7687"
AUTH = ("neo4j", "testpassword")
driver = GraphDatabase.driver(URI, auth=AUTH)

def run(cypher, **params):
    with driver.session() as s:
        return list(s.run(cypher, **params))

print(run("RETURN 'connected' AS status")[0]["status"])

## 적재 — MERGE로 멱등하게

In [ ]:
run("""
MERGE (c:Customer {id: 1, name: '이수진', tier: 'gold'})
MERGE (o:Order {id: 101, total: 52000})
MERGE (p1:Product {id: 'A', name: '키보드'})
MERGE (p2:Product {id: 'B', name: '마우스'})
MERGE (c)-[:PLACES]->(o)
MERGE (o)-[:CONTAINS {qty: 1, price: 40000}]->(p1)
MERGE (o)-[:CONTAINS {qty: 1, price: 12000}]->(p2)
""")
print('loaded')

## 멀티홉 조회 — 벡터 RAG가 못 풀던 질문

In [ ]:
rows = run("""
MATCH (c:Customer {name: '이수진'})-[:PLACES]->(:Order)-[:CONTAINS]->(p:Product)
RETURN p.name AS product
""")
print([r["product"] for r in rows])

## 연습

1. Ch 4의 의료 온톨로지(Drug/Disease/Ingredient)를 적재하라.
2. "두통을 치료하는 약 중 아세틸살리실산을 포함한 것"을 한 질의로 찾아라.
3. "같은 상품을 산 다른 고객" 3홉 추천 쿼리를 작성하라.